# exp150_formation_physical_imputer_revisit train

Fold-safe audit of a hidden-safe formation contact physical imputer. This notebook does not train boosters and does not create a submission.

## Contents

1. Setup and configuration
2. Data preview
3. Fold-safe physical imputer audit
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from settings import ExperimentPaths
from formation_physical_imputer_revisit import FORMATION_COLUMNS, METHODS, CANDIDATES, load_wells, run_audit

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = paths.config

print('experiment:', config['experiment']['name'])
print('route:', config['experiment']['route'])
print('parent:', config['lineage']['parent'])
print('formation columns:', FORMATION_COLUMNS)
print('surface methods:', METHODS)
print('physical candidates:', CANDIDATES)
print('train dir:', paths.train_data_dir)
print('artifacts dir:', paths.artifacts_dir)
print('features dir:', paths.features_dir)
print('metrics path:', paths.metrics_path)


## 2. Data preview

In [ ]:
wells_preview = load_wells(paths.train_data_dir, max_wells=5)
preview_rows = []
for well in wells_preview:
    frame = well.frame
    preview_rows.append({
        'well': well.well,
        'rows': len(frame),
        'known_prefix_rows': int(frame['TVT_input'].notna().sum()),
        'eval_rows': int(frame['TVT_input'].isna().sum()),
        'anchor_row': int(well.anchor_position),
        'formation_missing_total': int(frame[list(FORMATION_COLUMNS)].isna().sum().sum()),
    })
pd.DataFrame(preview_rows)


## 3. Fold-safe physical imputer audit

In [ ]:
metrics = run_audit(
    train_dir=paths.train_data_dir,
    artifacts_dir=paths.artifacts_dir,
    features_dir=paths.features_dir,
    metrics_path=paths.metrics_path,
    config=config,
    debug=False,
)
print(json.dumps(metrics['best_candidate'], indent=2, ensure_ascii=False))
print('score rows:', metrics['score_rows'])


## 4. Metrics and artifacts

In [ ]:
candidate_metrics = pd.read_csv(paths.artifacts_dir / 'candidate_metrics.csv')
distance_metrics = pd.read_csv(paths.artifacts_dir / 'distance_bucket_metrics.csv')
confidence_metrics = pd.read_csv(paths.artifacts_dir / 'confidence_bucket_metrics.csv')
surface_metrics = pd.read_csv(paths.artifacts_dir / 'surface_proxy_metrics.csv')

display(candidate_metrics.sort_values(['rmse', 'well_rmse_max']).head(12))
display(distance_metrics.sort_values(['method', 'candidate', 'bucket']).head(18))
display(confidence_metrics.sort_values(['method', 'candidate', 'confidence_feature', 'bucket']).head(18))
display(surface_metrics)

print('OOF features:', paths.features_dir / 'formation_physical_oof_features.csv')
print('candidate metrics:', paths.artifacts_dir / 'candidate_metrics.csv')
print('distance bucket metrics:', paths.artifacts_dir / 'distance_bucket_metrics.csv')
print('confidence bucket metrics:', paths.artifacts_dir / 'confidence_bucket_metrics.csv')
print('surface proxy metrics:', paths.artifacts_dir / 'surface_proxy_metrics.csv')
print('prefix calibration:', paths.artifacts_dir / 'formation_prefix_calibration.csv')
